In [ ]:
# import os
# os.environ["CUDA_VISIBLE_DEVICES"]="3"

import csv
import array
import base64
import xmltodict
import pylab as plt
from glob import glob
from tqdm import tqdm
import numpy as np
import pandas as pd
import pylab as plt
import seaborn as sns
from scipy.stats import chi2_contingency
from scipy.stats import chi2_contingency, ttest_ind
from sklearn.ensemble import RandomForestClassifier
import scipy.signal as signal
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import MultipleLocator
# Plotting settings
import scienceplots
import matplotlib
STYLE = ['science', 'nature']
import numpy as np

# class ECGXMLReader:
#     """ Extract voltage data from a ECG XML file """
#     def __init__(self, path, unknown_index=0 , augmentLeads=False):
#         # try: 
#         with open(path, 'rb') as xml:
#             self.ECG = xmltodict.parse(xml.read().decode('utf8'))

#         self.unknown_index          = unknown_index
#         self.augmentLeads           = augmentLeads
#         self.path                   = path

#         self.PatientDemographics    = self.ECG['RestingECG']['PatientDemographics']
#         self.TestDemographics       = self.ECG['RestingECG']['TestDemographics']
#         self.RestingECGMeasurements = self.ECG['RestingECG']['RestingECGMeasurements']
#         self.Waveforms              = self.ECG['RestingECG']['Waveform']

#         self.LeadVoltages           = self.makeLeadVoltages()
        
#         # except Exception as e:
#         #     print(str(e))
    
#     def makeLeadVoltages(self):

#         num_leads = 0

#         leads = {}

#         for lead in self.Waveforms[self.unknown_index]['LeadData']:
#             num_leads += 1
            
#             lead_data = lead['WaveFormData']
#             lead_b64  = base64.b64decode(lead_data)
#             lead_vals = np.array(array.array('h', lead_b64))

#             leads[ lead['LeadID'] ] = lead_vals
        
#         if num_leads == 8 and self.augmentLeads:

#             leads['III'] = np.subtract(leads['II'], leads['I'])
#             leads['aVR'] = np.add(leads['I'], leads['II'])*(-0.5)
#             leads['aVL'] = np.subtract(leads['I'], 0.5*leads['II'])
#             leads['aVF'] = np.subtract(leads['II'], 0.5*leads['I'])
        
#         return leads

#     def getLeadVoltages(self, LeadID):
#         return self.LeadVoltages[LeadID]
    
#     def getAllVoltages(self):
#         return self.LeadVoltages

class ECGXMLReader:
    """ Extract voltage data from a ECG XML file """
    def __init__(self, path, unknown_index=0 , augmentLeads=False):
        with open(path, 'rb') as xml:
            self.ECG = xmltodict.parse(xml.read().decode('utf8'))

        self.unknown_index          = unknown_index
        self.augmentLeads           = augmentLeads
        self.path                   = path

        self.PatientDemographics    = self.ECG['RestingECG']['PatientDemographics']
        self.TestDemographics       = self.ECG['RestingECG']['TestDemographics']
        self.RestingECGMeasurements = self.ECG['RestingECG']['RestingECGMeasurements']
        self.Waveforms              = self.ECG['RestingECG']['Waveform']

        self.LeadVoltages           = self.makeLeadVoltages()
    
    def makeLeadVoltages(self):
        num_leads = 0
        leads = {}
        for lead in self.Waveforms[self.unknown_index]['LeadData']:
            num_leads += 1
            lead_data = lead['WaveFormData']
            lead_b64  = base64.b64decode(lead_data)
            lead_vals = np.array(array.array('h', lead_b64))
            leads[lead['LeadID']] = lead_vals
        
        if num_leads == 8 and self.augmentLeads:
            leads['III'] = np.subtract(leads['II'], leads['I'])
            leads['aVR'] = np.add(leads['I'], leads['II'])*(-0.5)
            leads['aVL'] = np.subtract(leads['I'], 0.5*leads['II'])
            leads['aVF'] = np.subtract(leads['II'], 0.5*leads['I'])
        
        return leads

    def getLeadVoltages(self, LeadID):
        return self.LeadVoltages[LeadID]
    
    def getAllVoltages(self):
        return self.LeadVoltages

    def extract_p_wave(self, lead_id, fs=500):
        lead_data = self.getLeadVoltages(lead_id)
        
        # Bandpass filter to remove noise
        nyquist = 0.5 * fs
        low = 0.5 / nyquist
        high = 40 / nyquist
        b, a = signal.butter(1, [low, high], btype='band')
        filtered_lead = signal.filtfilt(b, a, lead_data)
        
        # Detect R-peaks
        r_peaks, _ = signal.find_peaks(filtered_lead, distance=fs/2.5, height=np.mean(filtered_lead) + 0.5*np.std(filtered_lead))
        
        # Extract P-wave segments
        p_wave_segments = []
        p_wave_window = int(0.2 * fs)  # 200 ms window before R-peak
        for r_peak in r_peaks:
            start = max(0, r_peak - p_wave_window)
            end = r_peak
            p_wave_segments.append(filtered_lead[start:end])
        
        return p_wave_segments


import numpy as np
from scipy.interpolate import interp1d
from scipy.signal import resample

def time_warp(x, sigma=0.01, knot=4):
    from scipy.interpolate import CubicSpline
    orig_steps = np.arange(x.shape[1])
    
    random_warps = np.random.normal(loc=1.0, scale=sigma, size=(x.shape[0], knot+2))
    warp_steps = (np.ones((x.shape[0],1))*(np.linspace(0, x.shape[1]-1., num=knot+2))).T
    
    ret = np.zeros_like(x)
    for i, pat in enumerate(x):
        warper = CubicSpline(warp_steps[:,i], warp_steps[:,i] * random_warps[i])
        warped = warper(orig_steps)
        ret[i] = np.interp(warped, orig_steps, pat)
    return ret

def amplitude_scale(x, sigma=0.1):
    scale_factor = np.random.normal(loc=1.0, scale=sigma, size=(x.shape[0], 1))
    return x * scale_factor

def add_baseline_wander(x, amplitude=0.05):
    t = np.linspace(0, 2*np.pi, x.shape[1])
    baseline = amplitude * np.sin(t)
    return x + baseline

def add_gaussian_noise(x, sigma=0.01):
    noise = np.random.normal(loc=0.0, scale=sigma, size=x.shape)
    return x + noise

def random_permutation(x, max_segments=2):
    orig_steps = np.arange(x.shape[1])
    
    ret = np.zeros_like(x)
    for i, pat in enumerate(x):
        num_segments = np.random.randint(2, max_segments+1)
        splits = np.sort(np.random.choice(x.shape[1], num_segments-1, replace=False))
        splits = np.concatenate(([0], splits, [x.shape[1]]))
        segments = [pat[splits[j]:splits[j+1]] for j in range(num_segments)]
        np.random.shuffle(segments)
        ret[i] = np.concatenate(segments)
    
    return ret

def random_shift(x, max_shift=4):
    ret = np.zeros_like(x)
    shift = np.random.randint(-max_shift, max_shift)
    for i, pat in enumerate(x):
        ret[i] = np.roll(pat, shift, axis=0)
    return ret

# Function to apply all augmentations
def augment_ecg(ecg_data, aug_p=0.2, num_augmented=1):
    augmented_data = []
    for _ in range(num_augmented):
        aug_data = ecg_data.copy()
        if np.random.uniform()<aug_p: aug_data = time_warp(aug_data)
        if np.random.uniform()<aug_p: aug_data = amplitude_scale(aug_data)
        if np.random.uniform()<aug_p: aug_data = add_baseline_wander(aug_data)
        if np.random.uniform()<aug_p: aug_data = add_gaussian_noise(aug_data)
        if np.random.uniform()<aug_p: aug_data = random_shift(aug_data)
        # if np.random.uniform()>0.0: aug_data = random_permutation(aug_data)
        augmented_data.append(aug_data)
    return np.concatenate(augmented_data, axis=0)

def ecg_vis(fil,p_wave=False,figset=None,wave_color = '#000000'):
    ecg = ECGXMLReader(fil, unknown_index=0, augmentLeads=True)
    leads = ecg.makeLeadVoltages()
    
    # Set up the figure and axes
    if figset is None:
        fig, axs = plt.subplots(6, 2, figsize=(10, 7.5))
        fig.patch.set_facecolor('#FFF0F5')  # Light pink background for the entire figure
    else:
        fig, axs = figset
    
    # Define colors
    grid_color = '#FF69B4'  # Hot pink for the grid
    
    for i_, l_ in enumerate(leads.keys()):
        ax = axs[i_ % 6, i_ // 6]
        
        # Set background color for each subplot
        ax.set_facecolor('#FFF0F5')  # Light pink background
        
        # Plot the ECG wave
        if p_wave:
            data_ = ecg.extract_p_wave(l_, fs=500)[0]
        else:
            data_ = leads[l_]
        # print(data_[0])
        # assert 0
        ax.plot(data_, color=wave_color, linewidth=0.8)
        
        # Set up the grid
        ax.grid(True, which='both', color=grid_color, linestyle='-', linewidth=0.5, alpha=0.7)
        ymin,ymax = ax.get_ylim()
        dy = (ymax-ymin)/10
        # ax.xaxis.set_major_locator(MultipleLocator(100))
        ax.xaxis.set_minor_locator(MultipleLocator(20))
        # ax.yaxis.set_major_locator(MultipleLocator(100))
        ax.yaxis.set_minor_locator(MultipleLocator(dy))
        
        # Set up the axes
        ax.set_ylabel(l_, fontweight='bold', fontsize=12)
        if p_wave:
            ax.set_xlim(0, 100)
        else:
            ax.set_xlim(0, 600)
        # ax.set_ylim(-200,200)  # Adjust as needed
        
        if i_ != 11 and i_ != 5:
            ax.set_xticks([])
        else:
            ax.set_xlabel('Time (ms)', fontweight='bold', fontsize=12)
            ax.tick_params(axis='x', which='both', labelsize=12)
        
        ax.set_yticks([])
    
    plt.subplots_adjust(wspace=0.1, hspace=0.3)
    
    # Add a title to the figure
    fig.suptitle('12-Lead ECG', fontsize=18, fontweight='bold', y=0.98)
    
    # Add some general information (replace with actual data if available)
    # fig.text(0.05, 0.05, 'Patient ID: 123456\nDate: 2023-11-11\nHeart Rate: 75 bpm', fontsize=12)
    
    plt.tight_layout()
    return fig, axs

# Results

In [ ]:
import numpy as np
import pandas as pd
from glob import glob
import seaborn as sns
import pylab as plt
%matplotlib inline

In [ ]:
df_res = pd.DataFrame(columns=['com_arch','n_comp','num_heads','ecg_app','aug_p','n_inc','n_comp_ecg','n_comp_ehr','initial_learning_rate']+\
                     ['AUROC','Precision','Sensitivity','Specificity','Accuracy','F1 score']+\
                     ['AUROC_std','Precision_std','Sensitivity_std','Specificity_std','Accuracy_std','F1 score_std'])
idf_ = 0
fils = glob('res/*.npy')
for fil in fils:
    params_ = np.load(fil)
    res_ = pd.read_csv(fil.replace('npy','csv')).drop(columns=['Unnamed: 0'])
    df_res.loc[idf_] = list(params_)+list(res_.mean().values)+list(res_.std().values)
    idf_ += 1

df_res_rand = df_res[df_res['ecg_app']==-1]
df_res = df_res[df_res['ecg_app']!=-1]

In [ ]:
df_res.max()

In [ ]:
df_res.shape

In [ ]:
df_res.sort_values(by='AUROC',ascending=0).head(1)

In [ ]:
df_res.sort_values(by='F1 score',ascending=0).head(1)

In [ ]:
df_res.sort_values(by='Sensitivity',ascending=0).head(1)

In [ ]:
df_res.sort_values(by='Specificity',ascending=0).head(1)

In [ ]:
com_arch = 0
n_comp = 8
num_heads = 4
ecg_app = 0
aug_p = 0.5
n_inc = 5
n_comp_ecg = 16
n_comp_ehr = 16
initial_learning_rate = 0.0001

df_res[
(df_res['com_arch']==0)&\
(df_res['n_comp']==8)&\
(df_res['num_heads']==4)&\
(df_res['ecg_app']==0)&\
(df_res['aug_p']==0.5)&\
(df_res['n_inc']==5)&\
(df_res['n_comp_ecg']==16)&\
(df_res['n_comp_ehr']==16)&\
(df_res['initial_learning_rate']==0.0001)
]

In [ ]:
df_res[['com_arch','n_comp','num_heads','ecg_app','aug_p','n_inc','n_comp_ecg','n_comp_ehr','initial_learning_rate']].drop_duplicates().shape

In [ ]:
_ = []
for metr_ in ['AUROC','Precision','Sensitivity','Specificity','Accuracy','F1 score']:
    __ = df_res.sort_values(by=metr_,ascending=0)[['com_arch','n_comp','num_heads','ecg_app','aug_p','n_inc','n_comp_ecg','n_comp_ehr','initial_learning_rate']]
    # __ = __[__!=1]
    _.append( __.drop_duplicates().head(5) )

pd.concat(_).drop_duplicates().reset_index(drop=1)#.to_csv('best_configs.csv')

In [ ]:
# com_arch = int(argv[1])
# n_comp = int(argv[2]) 
# num_heads = int(argv[3]) 
# ecg_app = int(argv[4]) 
# aug_p = float(argv[5]) 
# n_inc = int(argv[6]) 
# n_comp_ecg = int(argv[7]) 
# n_comp_ehr = int(argv[8]) 
# initial_learning_rate = float(argv[9])

In [ ]:
# df_res.to_csv('res.csv')

## an example of the data

In [ ]:
# Assuming ECGXMLReader is imported from your custom module
fil = sorted(glob('../ECGs/*.XML'))[0]
print(fil)
with plt.style.context(STYLE):
    fig, axs = plt.subplots(6, 2, figsize=(6, 4))
    fig.patch.set_facecolor('#FFF0F5')  # Light pink background for the entire figure
    fig, axs = ecg_vis(fil,figset=(fig, axs))
    fig.suptitle('12-Lead ECG', fontsize=12, fontweight='bold', y=0.96)
    name = 'ECG_example'
    fig.savefig(f'figs/{name}.pdf')
    fig.savefig(f'figs/{name}.svg')
    fig.savefig(f'figs/{name}.jpg', dpi=300)

## Overal metric distribution

In [ ]:
# Assuming your CSV data is loaded into a DataFrame called 'df'
metrics = ['AUROC', 'Precision', 'Sensitivity', 'Specificity', 'Accuracy', 'F1 score']

with plt.style.context(STYLE):
    fig,ax = plt.subplots(figsize=(5,3))
    sns.boxplot(data=df_res[metrics])
    plt.title('Performance Metrics Distribution',fontsize=14)
    plt.ylabel('Score',fontsize=14)
    plt.xticks(rotation=45,fontsize=12)
    plt.yticks(fontsize=12)
    plt.tight_layout()
    name = 'all_metrics'
    fig.savefig(f'figs/{name}.pdf')
    fig.savefig(f'figs/{name}.svg')
    fig.savefig(f'figs/{name}.jpg', dpi=300)

## Best AUC model spider plot

In [ ]:
# Select the best performing model (highest AUROC)
best_model = df_res.loc[df_res['AUROC'].idxmax()]

metrics = ['AUROC', 'Precision', 'Sensitivity', 'Specificity', 'Accuracy', 'F1 score']
values = best_model[metrics].values

angles = np.linspace(0, 2*np.pi, len(metrics), endpoint=False)
values = np.concatenate((values, [values[0]]))
angles = np.concatenate((angles, [angles[0]]))

with plt.style.context(STYLE):
    fig, ax = plt.subplots(figsize=(4, 4), subplot_kw=dict(projection='polar'))
    ax.plot(angles, values)
    ax.fill(angles, values, alpha=0.25)
    ax.set_thetagrids(angles[:-1] * 180/np.pi, metrics)
    ax.set_yticks([0.2, 0.4, 0.6, 0.8])  # Example: setting y-ticks at 0.2 intervals
    ax.set_rlabel_position(40)  # Adjust this value as needed
    
    ax.tick_params(axis='x', which='major', pad=18)
    plt.xticks(fontsize=12)    
    plt.yticks(fontsize=11)
    # ax.set_title("Best Model Performance Metrics",fontsize=14)
    name = 'polar-best'
    fig.savefig(f'figs/{name}.pdf')
    fig.savefig(f'figs/{name}.svg')
    fig.savefig(f'figs/{name}.jpg', dpi=300)

## Impact of augmentation

In [ ]:
aug_impact = df_res.groupby('aug_p')[metrics].mean()
aug_impact

## impact of how to compress ECG in deep learning model

In [ ]:
ecg_app_impact = df_res.groupby('ecg_app')[metrics].mean()
ecg_app_impact

## Best AUROC

In [ ]:
df_res.sort_values(by='AUROC',ascending=0).head(1)#.round(2).to_csv()

## Best Sensitivity

In [ ]:
df_res.sort_values(by='Sensitivity',ascending=0).head(1)#.round(2).to_csv()

## Best Specificity

In [ ]:
df_res.sort_values(by='Specificity',ascending=0).head(1)#.round(2).to_csv()

## Mean and statdard deviation of the metrics

In [ ]:
df_res_rand[['AUROC','Precision','Sensitivity','Specificity','Accuracy','F1 score']].mean().to_frame()

In [ ]:
df_res_rand[['AUROC','Precision','Sensitivity','Specificity','Accuracy','F1 score']].std().to_frame()

## Max of the metrics

In [ ]:
bs_df = df_res[['AUROC','Precision','Sensitivity','Specificity','Accuracy','F1 score']].idxmax()
for metr_ in ['AUROC','Precision','Sensitivity','Specificity','Accuracy','F1 score']:
    idx = bs_df[metr_]
    bs_df_ = df_res.loc[idx]
    bs_df_ = bs_df_[[metr_,f'{metr_}_std']]
    m_,s_ = bs_df_.values
    print(f'{metr_}: {m_:2.2f} ({s_:2.2f})')

In [ ]:
df_res[['AUROC','Precision','Sensitivity','Specificity','Accuracy','F1 score']].max().to_frame()

## Max of the metrics when we do not include any information about EHR (only ECG)

In [ ]:
df_res[df_res['n_comp_ehr']==0][['AUROC','Precision','Sensitivity','Specificity','Accuracy','F1 score']].max().to_frame()

In [ ]:
bs_df = df_res[df_res['n_comp_ehr']==0][['AUROC','Precision','Sensitivity','Specificity','Accuracy','F1 score']].idxmax()
for metr_ in ['AUROC','Precision','Sensitivity','Specificity','Accuracy','F1 score']:
    idx = bs_df[metr_]
    bs_df_ = df_res.loc[idx]
    bs_df_ = bs_df_[[metr_,f'{metr_}_std']]
    m_,s_ = bs_df_.values
    print(f'{metr_}: {m_:2.2f} ({s_:2.2f})')

## Max of the metrics when we do not include any information about ECG (only EHR)

In [ ]:
df_res[df_res['n_comp_ecg']==0][['AUROC','Precision','Sensitivity','Specificity','Accuracy','F1 score']].max().to_frame()

In [ ]:
bs_df = df_res[df_res['n_comp_ecg']==0][['AUROC','Precision','Sensitivity','Specificity','Accuracy','F1 score']].idxmax()
for metr_ in ['AUROC','Precision','Sensitivity','Specificity','Accuracy','F1 score']:
    idx = bs_df[metr_]
    bs_df_ = df_res.loc[idx]
    bs_df_ = bs_df_[[metr_,f'{metr_}_std']]
    m_,s_ = bs_df_.values
    print(f'{metr_}: {m_:2.2f} ({s_:2.2f})')

## Max of the metrics when we include both ECG and EHR

In [ ]:
df_res[(df_res['n_comp_ecg']!=0) & (df_res['n_comp_ecg']!=0)][['AUROC','Precision','Sensitivity','Specificity','Accuracy','F1 score']].max().to_frame()

### This shown increase when we include both EHR and ECG means the importance of multimodal models

### Some more details about how different modelity conponent impact the metrics:

In [ ]:
df_res.groupby(['n_comp_ecg','n_comp_ehr'])[['AUROC','Precision','Sensitivity','Specificity','Accuracy','F1 score']].mean()

## Now we can make a figure to show how the modalities contribute in the metrics

In [ ]:
col_name = 'EHR contribution (\%)'
norm_ = df_res[['n_comp_ecg','n_comp_ehr']].sum(axis=1)
df_res['n_comp_ecg']/norm_+df_res['n_comp_ehr']/norm_
df_res[col_name] = df_res['n_comp_ehr']/norm_
df_res[col_name] = df_res[col_name].apply(lambda x:f'{100*x:2.0f}')

In [ ]:
df_res_ = df_res[[col_name,'AUROC','Precision','Sensitivity','Specificity','Accuracy','F1 score']]
# sns.scatterplot(data=df_res_,x=col_name,y='AUROC', palette='coolwarm')

In [ ]:
__

In [ ]:
with plt.style.context(STYLE):
    fig, axs = plt.subplots(2,2,figsize=(7,5))
    order = sorted([i for i in sorted(df_res_[col_name].unique())], key=lambda x: int(x.strip()))
    print(order)
    positions = [float(i) for i in sorted(df_res_[col_name].unique())]
    
    __ = df_res_rand[['AUROC','Precision','Sensitivity','Specificity','Accuracy','F1 score']].mean()

    print(df_res_[[col_name,'AUROC','Sensitivity','Specificity','F1 score']].groupby(col_name).max().idxmax())
    
    # sns.boxenplot(data=df_res_,x=col_name,y='AUROC', ax=axs[0,0], order=order)
    # axs[0,0].plot(np.arange(-1,9),10*[__['AUROC']],'k--',alpha=0.5)
    # axs[0,0].set_xlim(-0.5,7.5)
    # axs[0,0].set_ylim(bottom=0.45)
    # axs[0,0].set_ylabel('AUROC',fontsize=13)
    
    sns.boxenplot(data=df_res_,x=col_name,y='Accuracy', ax=axs[0,0], order=order)
    axs[0,0].plot(np.arange(-1,9),10*[__['Accuracy']],'k--',alpha=0.5)
    axs[0,0].set_xlim(-0.5,7.5)
    axs[0,0].set_ylim(bottom=0.45)
    axs[0,0].set_ylabel('Accuracy',fontsize=13)
    
    sns.boxenplot(data=df_res_,x=col_name,y='Sensitivity', ax=axs[1,0], order=order)
    axs[1,0].plot(np.arange(-1,9),10*[__['Sensitivity']],'k--',alpha=0.5,label='Random prediction')
    axs[1,0].set_xlim(-0.5,7.5)
    # axs[1,0].set_ylim(bottom=0.45)
    axs[1,0].legend(fontsize=10)
    axs[1,0].set_ylabel('Sensitivity',fontsize=13)
    axs[1,0].set_xlabel('EHR contribution (\%)',fontsize=13)

    sns.boxenplot(data=df_res_,x=col_name,y='Specificity', ax=axs[0,1], order=order)
    axs[0,1].plot(np.arange(-1,9),10*[__['Specificity']],'k--',alpha=0.5)
    axs[0,1].set_xlim(-0.5,7.5)
    axs[0,1].set_ylim(bottom=0.6)
    axs[0,1].set_ylabel('Specificity',fontsize=13)
    
    sns.boxenplot(data=df_res_,x=col_name,y='F1 score', ax=axs[1,1], order=order)
    axs[1,1].plot(np.arange(-1,9),10*[__['F1 score']],'k--',alpha=0.5)
    axs[1,1].set_xlim(-0.5,7.5)
    # axs[1,1].set_ylim(bottom=0.45)
    axs[1,1].set_ylabel('F1 score',fontsize=13)
    axs[1,1].set_xlabel('EHR contribution (\%)',fontsize=13)
    
    axs[0,0].set_xlabel('')
    axs[0,1].set_xlabel('')
    # axs[0,0].set_xticks([])
    # axs[0,1].set_xticks([])
    axs[0,0].tick_params(axis='x', which='both', labelsize=0)
    axs[0,1].tick_params(axis='x', which='both', labelsize=0)
    axs[0,0].tick_params(axis='y', which='both', labelsize=12)
    axs[0,1].tick_params(axis='y', which='both', labelsize=12) 
    axs[1,0].tick_params(axis='both', which='both', labelsize=12)
    axs[1,1].tick_params(axis='both', which='both', labelsize=12)

    plt.subplots_adjust(hspace=0.1,wspace=0.25)
    name = 'modalities_impact'
    fig.savefig(f'figs/{name}.pdf')
    fig.savefig(f'figs/{name}.svg')
    fig.savefig(f'figs/{name}.jpg', dpi=300)

In [ ]:
import os, itertools, numpy as np, pandas as pd
from scipy.stats import mannwhitneyu

# Ensure output dir
out_dir = 'res_review'
os.makedirs(out_dir, exist_ok=True)

# Detect available metrics from your df_res_ (skip non-existing)
candidate_metrics = ['AUROC','Accuracy','Sensitivity','Specificity','F1 score']
metrics = [m for m in candidate_metrics if m in df_res_.columns]

# Build sorted group order like your plot
def _to_int_sort_key(x):
    try:
        return int(str(x).strip())
    except Exception:
        return x

groups = sorted(df_res_[col_name].dropna().unique(), key=_to_int_sort_key)

def cliffs_delta(x, y):
    # Cliff's delta: (sum_{i,j} sign(x_i - y_j)) / (n_x * n_y)
    x = np.asarray(x).reshape(-1)
    y = np.asarray(y).reshape(-1)
    nx, ny = len(x), len(y)
    if nx == 0 or ny == 0:
        return np.nan
    # Efficient approximation using ranks could be used; direct method is fine at these sizes
    count = 0
    for xi in x:
        count += np.sum(xi > y) - np.sum(xi < y)
    return count / (nx * ny)

def bh_adjust(pvals):
    p = np.array(pvals, dtype=float)
    n = p.size
    order = np.argsort(p)
    ranks = np.empty_like(order)
    ranks[order] = np.arange(1, n+1)
    adj = p * n / ranks
    # monotonicity
    adj_sorted = np.minimum.accumulate(adj[order][::-1])[::-1]
    out = np.empty_like(adj_sorted)
    out[order] = np.minimum(adj_sorted, 1.0)
    return out

all_rows = []

for metric in metrics:
    # Collect p-values first for BH
    tmp_rows = []
    pvals = []
    pairs = []
    for g1, g2 in itertools.combinations(groups, 2):
        x = df_res_.loc[df_res_[col_name] == g1, metric].dropna().values
        y = df_res_.loc[df_res_[col_name] == g2, metric].dropna().values
        if len(x) == 0 or len(y) == 0:
            stat, pval = np.nan, np.nan
            delta = np.nan
        else:
            stat, pval = mannwhitneyu(x, y, alternative='two-sided')
            delta = cliffs_delta(x, y)
        row = {
            'metric': metric,
            'group1': g1,
            'group2': g2,
            'n1': len(x),
            'n2': len(y),
            'median1': np.median(x) if len(x) else np.nan,
            'median2': np.median(y) if len(y) else np.nan,
            'mean1': np.mean(x) if len(x) else np.nan,
            'mean2': np.mean(y) if len(y) else np.nan,
            'mannwhitney_u': stat,
            'p_value': pval,
            "cliffs_delta": delta,
        }
        tmp_rows.append(row)
        pvals.append(pval if np.isfinite(pval) else 1.0)
        pairs.append((g1, g2))

    if tmp_rows:
        padj = bh_adjust(pvals)
        for i, row in enumerate(tmp_rows):
            row['p_adj_bh'] = float(padj[i])
            # significance stars
            p = row['p_adj_bh']
            if not np.isfinite(p):
                stars = ''
            elif p < 0.001:
                stars = '***'
            elif p < 0.01:
                stars = '**'
            elif p < 0.05:
                stars = '*'
            else:
                stars = ''
            row['significant'] = stars
        all_rows.extend(tmp_rows)

stats_df = pd.DataFrame(all_rows)
stats_path = os.path.join(out_dir, 'fig3_pairwise_stats.csv')
stats_df.to_csv(stats_path, index=False)
print(f'Saved pairwise stats: {stats_path}')

def fmt_p(p):
    if p is None or not np.isfinite(p):
        return ''
    return '<=0.05' if p <= 0.05 else f'{p:.2f}'
# Also save per-metric significance matrices (group x group with p_adj)
for metric in ['AUROC']: #metrics:
    dfm = stats_df[stats_df['metric'] == metric].copy()
    if dfm.empty:
        continue
    mat = pd.DataFrame(index=groups, columns=groups, dtype=float)
    for _, r in dfm.iterrows():
        g1, g2 = r['group1'], r['group2']
        mat.loc[g1, g2] = fmt_p(r['p_adj_bh'])
        mat.loc[g2, g1] = fmt_p(r['p_adj_bh'])
        mat.loc[g1, g1] = 1.0
        mat.loc[g2, g2] = 1.0
    safe_metric = metric.replace(' ', '_')
    mat_path = os.path.join(out_dir, f'fig3_{safe_metric}_padj_matrix.csv')
    mat.to_csv(mat_path)
    print(f'Saved {metric} p-adj matrix: {mat_path}')
mat

In [ ]:
df_res_

In [ ]:
import os, numpy as np, pandas as pd
from scipy.stats import mannwhitneyu, ttest_ind, ttest_rel, wilcoxon, kruskal

out_dir = 'res_review'
os.makedirs(out_dir, exist_ok=True)

# Set your group of interest
target_group = '33'  # change if needed, will match on string form
# Metrics present in df_res_
candidate_metrics = ['AUROC','Accuracy','Sensitivity','Specificity','F1 score']
metrics = [m for m in candidate_metrics if m in df_res_.columns]

def fmt_p(p):
    if p is None or not np.isfinite(p):
        return ''
    return '<=0.05' if p <= 0.05 else f'{p:.2f}'

# Helper to extract per-metric arrays for a given group value (robust to string/int labels)
def group_values(group_val):
    mask = df_res_[col_name].astype(str).str.strip() == str(group_val).strip()
    return {m: df_res_.loc[mask, m].dropna().values for m in metrics}

# Target and other groups
vals_target = group_values(target_group)
all_groups = df_res_[col_name].dropna().unique()
others = [g for g in all_groups if str(g).strip() != str(target_group).strip()]

# Sort others like the plot did (by integer if possible)
def sort_key(x):
    sx = str(x).strip()
    return (0, int(sx)) if sx.isdigit() else (1, sx)

rows = []
for g in sorted(others, key=sort_key):
    vals_other = group_values(g)
    row = {'comparison': f'{str(target_group).strip()} vs {str(g).strip()}'}
    for m in metrics:
        x = vals_target[m]
        y = vals_other[m]
        # Only compare the top ten for each group
        ntop = 18
        x_top = np.sort(x)[-ntop:] if len(x) > ntop else x
        y_top = np.sort(y)[-ntop:] if len(y) > ntop else y
        if len(x_top) == 0 or len(y_top) == 0:
            p = None
        else:
            # _, p = mannwhitneyu(x_top, y_top, use_continuity=True, alternative='two-sided')
            # _, p = ttest_rel(x_top, y_top)
            # _, p = ttest_ind(x_top, y_top, equal_var=False)
            _, p = wilcoxon(x_top, y_top)
            # _, p = kruskal(x, y)
        row[m] = fmt_p(p)
    rows.append(row)

focus_df = pd.DataFrame(rows)
focus_df = focus_df[['comparison'] + metrics]
display(focus_df)

csv_path = os.path.join(out_dir, f'fig3_focus_{str(target_group).strip()}_pvals.csv')
focus_df.to_csv(csv_path, index=False)
print('Saved:', csv_path)

In [ ]:
df_res_.head(2)

In [ ]:
__ss = df_res[[col_name,'Accuracy']].groupby(col_name).max().sort_values(by='Accuracy',ascending=False) 
__ss

In [ ]:
(__ss.loc['67']-__ss.loc[' 0'])/__ss.loc[' 0']*100, (__ss.loc['67']-__ss.loc['100'])/__ss.loc['100']*100

In [ ]:
df_res[[col_name,'Accuracy']].groupby(col_name).mean().sort_values(by='Accuracy',ascending=False) 

In [ ]:
df_res[[col_name,'Accuracy']].groupby(col_name).std().sort_values(by='Accuracy',ascending=False) 

## Now let's do a bit of a deeper research on the best results, meaning that if we select the best 10  models and run each for 10 times, what are the data points that are usually predicted "right" or "wrong" when they are in the test set

In [ ]:
nfiles = len(glob('res_best/*.npy'))
config_list = []
metric_list = []
pred_list = []
for path_ in [i.replace('.npy','') for i in glob('res_best/*.npy')]:
    config_list.append( np.load(f'{path_}.npy') )
    metric_list.append( pd.read_csv(f'{path_}.csv') )
    pred_list.append( pd.read_csv(f'{path_}_pred.csv') )
print(nfiles)

In [ ]:
pred_list[0].head(2)

In [ ]:
acc_mat = []
for i_ in range(nfiles):
    _ = ( pred_list[i_]['y_true']==pred_list[i_]['y_pred'].round(0) ).astype(int)
    acc_mat.append(_)
acc_mat = 100*np.array(acc_mat)/nfiles
acc_mat.shape

In [ ]:
df_ehr.columns[1:5].to_kist

In [ ]:
acc_mat = []
for i_ in range(nfiles):
    _ = ( pred_list[i_]['y_true']==pred_list[i_]['y_pred'].round(0) ).astype(int)
    acc_mat.append(_)
acc_mat = 100*np.array(acc_mat)/nfiles

df_ehr[['Age @ Index Stroke\n\n(year)', 'Sex \n\n1:M  \n2:F',
       'Race \n\n1:White 2:Black 3:Asian  4:Others',
       'Ethnicity \n\n1:Hispanic\n2:non-Hispanic)']]

### Here we can see how many percent of the time the prediction was "right"

In [ ]:
def_pred_prec = pd.DataFrame(index=np.arange(189),columns=['percent true'],data=np.sum(acc_mat,axis=0))
def_pred_prec

In [ ]:
df_ehr = pd.read_csv('ready_EHR.csv')
df_ehr.shape

In [ ]:
pd.read_csv('ready_Y.csv').value_counts()

In [ ]:
pd.concat([df_ehr,pd.read_csv('ready_Y.csv')],axis=1).columns[:5]

In [ ]:
pd.concat([df_ehr,pd.read_csv('ready_Y.csv')],axis=1)[[
    'Age @ Index Stroke\n\n(year)',
    'Sex \n\n1:M  \n2:F',
     'Race \n\n1:White 2:Black 3:Asian  4:Others',
     'Ethnicity \n\n1:Hispanic\n2:non-Hispanic)',
     'Number of months the heart was monitored',
     'Diagnsois of afib \n\n1: No\n2: Yes\n',
     ]]

In [ ]:
def_pred_prec.shape,df_ehr.shape

In [ ]:
pd.concat([def_pred_prec,df_ehr[['HEMOGLOBIN @ Index Stroke']]],axis=1).corr()

### Lets get the 5 patients with best (most frequently predicted right) and worse (most frequently predicted wrong)

In [ ]:
nsample = 5

In [ ]:
# def_pred_prec.sort_values('percent true').iloc[:nsample]

In [ ]:
def_pred_prec.sort_values('percent true').iloc[:nsample].index.to_list()

In [ ]:
df_ehr_worse = df_ehr.loc[def_pred_prec.sort_values('percent true').iloc[:nsample].index.to_list()]
df_ehr_worse

In [ ]:
for i_mrn,mnr in enumerate(sorted([p.split('/')[-1].replace('.XML','') for p in glob('../ECGs/*.XML')])[:5]):
    # Assuming ECGXMLReader is imported from your custom module
    fil = f'../ECGs/{mnr}.XML'
    print(fil)
    with plt.style.context(STYLE):
        fig, axs = plt.subplots(6, 2, figsize=(6, 4))
        fig.patch.set_facecolor('#FFF0F5')  # Light pink background for the entire figure
        fig, axs = ecg_vis(fil,figset=(fig, axs))
        fig.suptitle('12-Lead ECG', fontsize=12, fontweight='bold', y=0.96)
        name = f'ECG_example_worse_{i_mrn}'
        fig.savefig(f'figs/{name}.pdf')
        fig.savefig(f'figs/{name}.svg')
        fig.savefig(f'figs/{name}.jpg', dpi=300)

In [ ]:
def_pred_prec.sort_values('percent true',ascending=0).iloc[:nsample]

In [ ]:
def_pred_prec.sort_values('percent true',ascending=0).iloc[:nsample].index.to_list()

In [ ]:
df_ehr_best = df_ehr.loc[def_pred_prec.sort_values('percent true').iloc[-nsample:].index.to_list()]
df_ehr_best

In [ ]:
for i_mrn,mnr in enumerate(sorted([p.split('/')[-1].replace('.XML','') for p in glob('../ECGs/*.XML')])[:5]):
    # Assuming ECGXMLReader is imported from your custom module
    fil = f'../ECGs/{mnr}.XML'
    print(fil)
    with plt.style.context(STYLE):
        fig, axs = plt.subplots(6, 2, figsize=(6, 4))
        fig.patch.set_facecolor('#FFF0F5')  # Light pink background for the entire figure
        fig, axs = ecg_vis(fil,figset=(fig, axs))
        fig.suptitle('12-Lead ECG', fontsize=12, fontweight='bold', y=0.96)
        name = f'ECG_example_best_{i_mrn}'
        fig.savefig(f'figs/{name}.pdf')
        fig.savefig(f'figs/{name}.svg')
        fig.savefig(f'figs/{name}.jpg', dpi=300)

In [ ]:
# cols_feats = df_ehr_worse.drop(columns=['MRN']).columns

In [ ]:
# nsample = 30
# df_ehr_worse = df_ehr.loc[def_pred_prec.sort_values('percent true').iloc[:nsample].index.to_list()]
# df_ehr_best = df_ehr.loc[def_pred_prec.sort_values('percent true').iloc[-nsample:].index.to_list()]

# data = pd.concat([df_ehr_worse.assign(c=0),df_ehr_best.assign(c=1)],axis=0)
# print(data.shape)
# data = data.merge(pd.read_csv('Afib_data_v2.csv')[['MRN','Diagnsois of afib \n\n1: No\n2: Yes\n']],on='MRN',how='left')
# print(data.shape)
# # data

In [ ]:
# from sklearn.ensemble import RandomForestClassifier
# # import matplotlib.pyplot as plt
# import seaborn as sns

# x = data.drop(columns=['MRN','c','Diagnsois of afib \n\n1: No\n2: Yes\n'])
# y = data['c']

# # Initialize a list to store feature importances from each iteration
# importances_list = []

# # Train the random forest classifier 10 times and store the feature importances
# for i in range(10):
#     rf = RandomForestClassifier(n_estimators=100)
#     rf.fit(x, y)
#     importances_list.append(rf.feature_importances_)

# # Convert the list of importances to a DataFrame
# importances_df = pd.DataFrame(importances_list, columns=x.columns)

# # Melt the DataFrame to long format for seaborn
# importances_long_df = importances_df.melt(var_name='Feature', value_name='Importance')

# # Calculate the mean importance for ordering
# mean_importances = importances_long_df.groupby('Feature')['Importance'].mean().sort_values(ascending=False)
# important_features = mean_importances.index.to_list()[:10]
# importances_long_df_ = importances_long_df[importances_long_df['Feature'].isin(important_features)]

# # Plot the feature importances with error bars using seaborn
# plt.figure(figsize=(6, 8))
# sns.barplot(x='Importance', y='Feature', data=importances_long_df_,
#             order=important_features)
# plt.title('Average Feature Importance with Error Bars')
# plt.xlabel('Importance')
# plt.ylabel('Feature')

In [ ]:
# _ = data[important_features[:1]+['c']].melt(id_vars=['c'])
# _['c'] = _['c'].replace({
#     0:'worst',
#     1:'best'
# })
# _.head(5)
# sns.boxenplot(data = _ , y='variable', x='value', hue='c')

## Now lets see if we can find out what is the most influencial variable in predictability

In [ ]:
x.columns

In [ ]:
from sklearn.ensemble import RandomForestRegressor
import seaborn as sns
import shap

# x = data.drop(columns=['MRN','Diagnsois of afib \n\n1: No\n2: Yes\n'])
# y = data['MRN']

x = df_ehr.drop(columns=['MRN','Number of months the heart was monitored'])
x.columns = [i.replace('\n',' ').split('1:No')[0].replace(' 1:M   2:F',
    '').replace('   1:White 2:Black 3:Asian  4:Others',''
    ).replace('   1:Hispanic 2:non-Hispanic)',''
    ).replace('  1:Never 2:Former 3: Current','').strip() for i in x.columns]
y = def_pred_prec['percent true']

# Initialize a list to store feature importances from each iteration
importances_list = []

# Train the random forest regressor 10 times and store the feature importances
for i in range(10):
    rf = RandomForestRegressor(n_estimators=100)
    rf.fit(x, y)
    importances_list.append(rf.feature_importances_)

# Convert the list of importances to a DataFrame
importances_df = pd.DataFrame(importances_list, columns=x.columns)

# Melt the DataFrame to long format for seaborn
importances_long_df = importances_df.melt(var_name='Feature', value_name='Importance')

# Calculate the mean importance for ordering
mean_importances = importances_long_df.groupby('Feature')['Importance'].mean().sort_values(ascending=False)
important_features = mean_importances.index.to_list()[:10]
importances_long_df_ = importances_long_df[importances_long_df['Feature'].isin(important_features)]

with plt.style.context(STYLE):
    fig, axs = plt.subplots(figsize=(3, 5))
    sns.barplot(x='Importance', y='Feature', data=importances_long_df_,
                order=important_features)
    plt.title('Average Feature Importance with Error Bars')
    plt.xlabel('Importance')
    plt.ylabel('Feature')
    name = f'importance_predictability'
    fig.savefig(f'figs/{name}.pdf')
    fig.savefig(f'figs/{name}.svg')
    fig.savefig(f'figs/{name}.jpg', dpi=300)

# --- SHAP waterfall plot for a single prediction ---

# Positional alignment
x_ = x.reset_index(drop=True)
y_ = y.reset_index(drop=True)

# Fit RF
rf = RandomForestRegressor(n_estimators=150, random_state=0)
rf.fit(x_, y_)

# SHAP using new API
explainer = shap.TreeExplainer(rf)
explanation = explainer(x_)  # returns shap.Explanation for all rows

# Pick instance by position
instance_idx = int(np.argmax(y_.to_numpy()))

# Plot with clean style and safe tick locators
with plt.style.context('default'):  # avoid custom STYLE here
    fig = plt.figure(figsize=(6, 4))
    shap.plots.waterfall(explanation[instance_idx], max_display=7, show=False)
    ax = plt.gca()
    from matplotlib.ticker import MaxNLocator, NullLocator
    ax.xaxis.set_major_locator(MaxNLocator(nbins=6))
    ax.xaxis.set_minor_locator(NullLocator())
    plt.tight_layout()
    name = 'shap_waterfall_predictability'
    fig.savefig(f'figs/{name}.pdf')
    fig.savefig(f'figs/{name}.svg')
    fig.savefig(f'figs/{name}.jpg', dpi=300)

In [ ]:
df_ehr = pd.read_csv('ready_EHR.csv')
# Create descriptive table (Table 1) for the paper
# Load and combine EHR and outcome data
df_combined = pd.concat([df_ehr, pd.read_csv('ready_Y.csv')], axis=1)

# Clean column names for better readability
# df_combined.columns = df_combined.columns.str.replace('\n\n', ' ')

demographic_dic = {
    'Age @ Index Stroke\n\n(year)': 'Age @ Index Stroke (year)',
    'Sex \n\n1:M  \n2:F': 'Sex 1:M  2:F',
    'Race \n\n1:White 2:Black 3:Asian  4:Others': 'Race 1:White 2:Black 3:Asian  4:Others',
    'Ethnicity \n\n1:Hispanic\n2:non-Hispanic)': 'Ethnicity 1:Hispanic 2:non-Hispanic)',
    'Number of months the heart was monitored': 'Number of months the heart was monitored',
    'Diagnsois of afib \n\n1: No\n2: Yes\n':'Diagnsois of afib 1: No 2: Yes',
}


# Define key variables for analysis
demographic_vars = [
    'Age @ Index Stroke (year)',
    'Sex 1:M  2:F',
    'Race 1:White 2:Black 3:Asian  4:Others',
    'Ethnicity 1:Hispanic 2:non-Hispanic)',
    'Number of months the heart was monitored',
    'Diagnsois of afib 1: No 2: Yes',
]

df_combined.columns = df_combined.columns.map(demographic_dic)
# Select demographic variables
df_demo = df_combined[demographic_vars].copy()

print("Dataset Overview:")
print(f"Total sample size: {len(df_demo)}")
print(f"AFib cases: {df_demo['Diagnsois of afib 1: No 2: Yes'].value_counts()}")
print("\nColumn names:")
for col in df_demo.columns:
    print(f"- {col}")


In [ ]:
# Create publication-ready descriptive table
def create_descriptive_table(df, outcome_col):
    """Create a descriptive statistics table stratified by outcome"""
    
    # Split data by outcome
    no_afib = df[df[outcome_col] == 0]  # No AFib
    yes_afib = df[df[outcome_col] == 1]  # Yes AFib
    
    results = []
    
    # Age (continuous variable)
    age_col = 'Age @ Index Stroke (year)'
    results.append({
        'Variable': 'Age (years)',
        'Total (N=189)': f"{df[age_col].mean():.1f} ± {df[age_col].std():.1f}",
        'No AFib (N={})'.format(len(no_afib)): f"{no_afib[age_col].mean():.1f} ± {no_afib[age_col].std():.1f}",
        'AFib (N={})'.format(len(yes_afib)): f"{yes_afib[age_col].mean():.1f} ± {yes_afib[age_col].std():.1f}",
        'P-value': f"{ttest_ind(no_afib[age_col].dropna(), yes_afib[age_col].dropna())[1]:.3f}"
    })
    
    # Monitoring duration (continuous variable)
    monitor_col = 'Number of months the heart was monitored'
    results.append({
        'Variable': 'Monitoring duration (months)',
        'Total (N=189)': f"{df[monitor_col].mean():.1f} ± {df[monitor_col].std():.1f}",
        'No AFib (N={})'.format(len(no_afib)): f"{no_afib[monitor_col].mean():.1f} ± {no_afib[monitor_col].std():.1f}",
        'AFib (N={})'.format(len(yes_afib)): f"{yes_afib[monitor_col].mean():.1f} ± {yes_afib[monitor_col].std():.1f}",
        'P-value': f"{ttest_ind(no_afib[monitor_col].dropna(), yes_afib[monitor_col].dropna())[1]:.3f}"
    })
    
    # Sex (categorical variable)
    sex_col = 'Sex 1:M  2:F'
    # Create contingency table for chi-square test
    sex_crosstab = pd.crosstab(df[sex_col], df[outcome_col])
    chi2_stat, p_val = chi2_contingency(sex_crosstab)[:2]
    
    results.append({
        'Variable': 'Sex, n (%)',
        'Total (N=189)': '',
        'No AFib (N={})'.format(len(no_afib)): '',
        'AFib (N={})'.format(len(yes_afib)): '',
        'P-value': f"{p_val:.3f}"
    })
    
    # Male
    male_total = len(df[df[sex_col] == 1])
    male_no_afib = len(no_afib[no_afib[sex_col] == 1])
    male_afib = len(yes_afib[yes_afib[sex_col] == 1])
    
    results.append({
        'Variable': '  Male',
        'Total (N=189)': f"{male_total} ({male_total/len(df)*100:.1f})",
        'No AFib (N={})'.format(len(no_afib)): f"{male_no_afib} ({male_no_afib/len(no_afib)*100:.1f})" if len(no_afib) > 0 else "0 (0.0)",
        'AFib (N={})'.format(len(yes_afib)): f"{male_afib} ({male_afib/len(yes_afib)*100:.1f})" if len(yes_afib) > 0 else "0 (0.0)",
        'P-value': ''
    })
    
    # Female
    female_total = len(df[df[sex_col] == 2])
    female_no_afib = len(no_afib[no_afib[sex_col] == 2])
    female_afib = len(yes_afib[yes_afib[sex_col] == 2])
    
    results.append({
        'Variable': '  Female',
        'Total (N=189)': f"{female_total} ({female_total/len(df)*100:.1f})",
        'No AFib (N={})'.format(len(no_afib)): f"{female_no_afib} ({female_no_afib/len(no_afib)*100:.1f})" if len(no_afib) > 0 else "0 (0.0)",
        'AFib (N={})'.format(len(yes_afib)): f"{female_afib} ({female_afib/len(yes_afib)*100:.1f})" if len(yes_afib) > 0 else "0 (0.0)",
        'P-value': ''
    })
    
    return pd.DataFrame(results)

# Generate the descriptive table
outcome_column = 'Diagnsois of afib 1: No 2: Yes'
descriptive_table = create_descriptive_table(df_demo, outcome_column)

print("Table 1. Baseline Characteristics of Study Participants")
print("=" * 80)
display(descriptive_table)


In [ ]:
# Extended descriptive table with all demographic variables
def create_comprehensive_table(df, outcome_col):
    """Create a comprehensive descriptive statistics table"""
    
    # Split data by outcome
    no_afib = df[df[outcome_col] == 0]  # No AFib
    yes_afib = df[df[outcome_col] == 1]  # Yes AFib
    
    results = []
    
    # Helper function for categorical variables
    def add_categorical_variable(var_col, var_name, categories_dict):
        # Chi-square test
        crosstab = pd.crosstab(df[var_col], df[outcome_col])
        chi2_stat, p_val = chi2_contingency(crosstab)[:2]
        
        # Header row
        results.append({
            'Variable': f'{var_name}, n (%)',
            'Total (N=189)': '',
            'No AFib (N={})'.format(len(no_afib)): '',
            'AFib (N={})'.format(len(yes_afib)): '',
            'P-value': f"{p_val:.3f}"
        })
        
        # Category rows
        for cat_value, cat_name in categories_dict.items():
            cat_total = len(df[df[var_col] == cat_value])
            cat_no_afib = len(no_afib[no_afib[var_col] == cat_value])
            cat_afib = len(yes_afib[yes_afib[var_col] == cat_value])
            
            results.append({
                'Variable': f'  {cat_name}',
                'Total (N=189)': f"{cat_total} ({cat_total/len(df)*100:.1f})",
                'No AFib (N={})'.format(len(no_afib)): f"{cat_no_afib} ({cat_no_afib/len(no_afib)*100:.1f})" if len(no_afib) > 0 else "0 (0.0)",
                'AFib (N={})'.format(len(yes_afib)): f"{cat_afib} ({cat_afib/len(yes_afib)*100:.1f})" if len(yes_afib) > 0 else "0 (0.0)",
                'P-value': ''
            })
    
    # Age (continuous variable)
    age_col = 'Age @ Index Stroke (year)'
    results.append({
        'Variable': 'Age (years), mean ± SD',
        'Total (N=189)': f"{df[age_col].mean():.1f} ± {df[age_col].std():.1f}",
        'No AFib (N={})'.format(len(no_afib)): f"{no_afib[age_col].mean():.1f} ± {no_afib[age_col].std():.1f}",
        'AFib (N={})'.format(len(yes_afib)): f"{yes_afib[age_col].mean():.1f} ± {yes_afib[age_col].std():.1f}",
        'P-value': f"{ttest_ind(no_afib[age_col].dropna(), yes_afib[age_col].dropna())[1]:.3f}"
    })
    
    # Sex
    sex_categories = {1: 'Male', 2: 'Female'}
    add_categorical_variable('Sex 1:M  2:F', 'Sex', sex_categories)
    
    # Race
    race_categories = {1: 'White', 2: 'Black', 3: 'Asian', 4: 'Others'}
    add_categorical_variable('Race 1:White 2:Black 3:Asian  4:Others', 'Race', race_categories)
    
    # Ethnicity
    ethnicity_categories = {1: 'Hispanic', 2: 'Non-Hispanic'}
    add_categorical_variable('Ethnicity 1:Hispanic 2:non-Hispanic)', 'Ethnicity', ethnicity_categories)
    
    # Monitoring duration (continuous variable)
    monitor_col = 'Number of months the heart was monitored'
    results.append({
        'Variable': 'Monitoring duration (months), mean ± SD',
        'Total (N=189)': f"{df[monitor_col].mean():.1f} ± {df[monitor_col].std():.1f}",
        'No AFib (N={})'.format(len(no_afib)): f"{no_afib[monitor_col].mean():.1f} ± {no_afib[monitor_col].std():.1f}",
        'AFib (N={})'.format(len(yes_afib)): f"{yes_afib[monitor_col].mean():.1f} ± {yes_afib[monitor_col].std():.1f}",
        'P-value': f"{ttest_ind(no_afib[monitor_col].dropna(), yes_afib[monitor_col].dropna())[1]:.3f}"
    })
    
    return pd.DataFrame(results)

# Generate comprehensive table
comprehensive_table = create_comprehensive_table(df_demo, outcome_column)

print("\\nTable 1. Baseline Characteristics of Study Participants by Atrial Fibrillation Status")
print("=" * 100)
display(comprehensive_table)

# Save to CSV for use in manuscript
comprehensive_table.to_csv('descriptive_table_afib.csv', index=False)
print("\\nTable saved as 'descriptive_table_afib.csv' for manuscript use.")


In [ ]:
# Create a formatted table suitable for publication
def format_table_for_publication(table_df):
    """Format the table for better publication appearance"""
    
    # Create a copy to avoid modifying original
    formatted_table = table_df.copy()
    
    # Clean up column names
    formatted_table.columns = [
        'Characteristic',
        'Total\\n(N=189)',
        'No AFib\\n(N=TBD)',  # Will be updated when we know actual counts
        'AFib\\n(N=TBD)',     # Will be updated when we know actual counts  
        'P-value'
    ]
    
    return formatted_table

# Format and display the publication-ready table
pub_table = format_table_for_publication(comprehensive_table)

print("PUBLICATION-READY TABLE")
print("Copy this output directly into your manuscript:")
print("=" * 100)

# Display with better formatting
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
display(pub_table)

# Also create a LaTeX-friendly version
print("\\n\\nLaTeX TABLE CODE:")
print("=" * 50)
print("% Copy this LaTeX code into your manuscript")
print("\\\\begin{table}[htbp]")
print("\\\\centering")
print("\\\\caption{Baseline Characteristics of Study Participants by Atrial Fibrillation Status}")
print("\\\\label{tab:baseline_characteristics}")
print("\\\\begin{tabular}{lcccc}")
print("\\\\hline")
print("Characteristic & Total (N=189) & No AFib & AFib & P-value \\\\\\\\")
print("\\\\hline")

for _, row in pub_table.iterrows():
    char = row['Characteristic'].replace('%', '\\\\%').replace('±', '$\\\\pm$')
    total = row['Total\\n(N=189)'].replace('±', '$\\\\pm$')
    no_afib = row['No AFib\\n(N=TBD)'].replace('±', '$\\\\pm$')
    afib = row['AFib\\n(N=TBD)'].replace('±', '$\\\\pm$')
    p_val = row['P-value']
    
    print(f"{char} & {total} & {no_afib} & {afib} & {p_val} \\\\\\\\")

print("\\\\hline")
print("\\\\end{tabular}")
print("\\\\footnotesize")
print("\\\\textit{Note: Continuous variables are presented as mean ± standard deviation.}")
print("\\\\textit{Categorical variables are presented as count (percentage).}")
print("\\\\textit{P-values calculated using t-tests for continuous variables and}")
print("\\\\textit{chi-square tests for categorical variables.}")
print("\\\\end{table}")

# Save formatted table
pub_table.to_csv('publication_table_afib.csv', index=False)
pub_table.to_excel('publication_table_afib.xlsx', index=False)
print("\\n\\nFormatted tables saved as:")
print("- publication_table_afib.csv")
print("- publication_table_afib.xlsx")


In [ ]:
# Additional statistics for the manuscript
print("ADDITIONAL STATISTICS FOR MANUSCRIPT")
print("=" * 60)

# Calculate AFib prevalence
afib_counts = df_demo[outcome_column].value_counts()
total_patients = len(df_demo)
afib_patients = afib_counts.get(1, 0)  # AFib = 2
no_afib_patients = afib_counts.get(0, 0)  # No AFib = 1

afib_prevalence = (afib_patients / total_patients) * 100

print(f"Total study population: {total_patients}")
print(f"Patients with AFib: {afib_patients} ({afib_prevalence:.1f}%)")
print(f"Patients without AFib: {no_afib_patients} ({100-afib_prevalence:.1f}%)")

# Age distribution
age_col = 'Age @ Index Stroke (year)'
print(f"\\nAge distribution:")
print(f"  Mean age: {df_demo[age_col].mean():.1f} ± {df_demo[age_col].std():.1f} years")
print(f"  Median age: {df_demo[age_col].median():.1f} years")
print(f"  Age range: {df_demo[age_col].min():.0f} - {df_demo[age_col].max():.0f} years")

# Monitoring duration
monitor_col = 'Number of months the heart was monitored'
print(f"\\nMonitoring duration:")
print(f"  Mean duration: {df_demo[monitor_col].mean():.1f} ± {df_demo[monitor_col].std():.1f} months")
print(f"  Median duration: {df_demo[monitor_col].median():.1f} months")
print(f"  Duration range: {df_demo[monitor_col].min():.1f} - {df_demo[monitor_col].max():.1f} months")

# Sex distribution
sex_col = 'Sex 1:M  2:F'
sex_counts = df_demo[sex_col].value_counts()
male_count = sex_counts.get(1, 0)
female_count = sex_counts.get(2, 0)

print(f"\\nSex distribution:")
print(f"  Male: {male_count} ({male_count/total_patients*100:.1f}%)")
print(f"  Female: {female_count} ({female_count/total_patients*100:.1f}%)")

# Race distribution
race_col = 'Race 1:White 2:Black 3:Asian  4:Others'
race_counts = df_demo[race_col].value_counts()
race_labels = {1: 'White', 2: 'Black', 3: 'Asian', 4: 'Others'}

print(f"\\nRace distribution:")
for race_code, count in race_counts.items():
    race_name = race_labels.get(race_code, f'Unknown ({race_code})')
    print(f"  {race_name}: {count} ({count/total_patients*100:.1f}%)")

# Ethnicity distribution
ethnicity_col = 'Ethnicity 1:Hispanic 2:non-Hispanic)'
ethnicity_counts = df_demo[ethnicity_col].value_counts()
ethnicity_labels = {1: 'Hispanic', 2: 'Non-Hispanic'}

print(f"\\nEthnicity distribution:")
for eth_code, count in ethnicity_counts.items():
    eth_name = ethnicity_labels.get(eth_code, f'Unknown ({eth_code})')
    print(f"  {eth_name}: {count} ({count/total_patients*100:.1f}%)")

print("\\n" + "=" * 60)
print("KEY FINDINGS FOR MANUSCRIPT:")
print("- Study includes {} patients with stroke".format(total_patients))
print("- AFib prevalence: {:.1f}%".format(afib_prevalence))
print("- Mean age: {:.1f} years".format(df_demo[age_col].mean()))
print("- Sex distribution: {:.1f}% male, {:.1f}% female".format(
    male_count/total_patients*100, female_count/total_patients*100))
print("- Mean monitoring duration: {:.1f} months".format(df_demo[monitor_col].mean()))


In [ ]:
from scipy.stats import pearsonr, kurtosis, skew, moment, normaltest, levene, ttest_ind, norm
def give_star(pv):
    if pv > 0.05:
        return '*'
    elif pv <= 0.05 and pv > 0.01:
        return '**'
    elif pv <= 0.01 and pv > 0.001:
        return '***'
    elif pv <= 0.001:
        return '****'

for i_ in [0,1,2]:
    g1 = x[important_features[i_]][y>50]
    g2 = x[important_features[i_]][y<50]
    
    _,p1 = levene(g1, g2)
    _,p_val = ttest_ind(g1, g2, equal_var=p1>0.05, nan_policy='omit')
    print(important_features[i_],np.round(p_val,2))

#### These results show although 
- Number of months the heart was monitored
- Age 
- HEMOGLOBIN
### are important, but their influence is not an individual influence since at least the the last two we can not see any significant difference while considering only the variable individually.

In [ ]:
!zip figs.zip -r figs/*